# P7 — E：改写攻击测试与结果汇总

**本人的工作**：整理 A–D 的历史测试成绩；在 D 提供的新生成器数据上准备改写测试集；调用四个已有检测器；保存逐条分数，计算攻击后指标并汇总。E 没有训练新的检测器。

**本次真实数据**：原始数据 449 篇，其中人类 150 篇、AI 299 篇。攻击后仍为 449 篇：150 篇人类原文保持不变，299 篇 AI 文本分别做了改写，AI 的真实标签仍为 1。

**流程**：原始数据 → 固定改写测试集 → A/B/C/D 对每篇文本打分 → 根据真实标签计算指标 → 与成员历史成绩并列展示 → 导出四个交付文件。

默认运行只读取已经完成的真实检测记录并重新计算指标，不下载模型、不读取 API key、不请求收费接口。最后的重新检测部分默认关闭。

## 1. 文件位置与运行方式

将配套包解压后，保留 `notebooks/` 和 `data/E/` 的相对位置。也可以把这两个文件夹合并到项目根目录。
本 notebook 从项目根目录或 `notebooks/` 目录启动均可。计算结果写入 `analysis/E_notebook_export/results/`，不会覆盖现有的正式 `results/` 或原来的检测进度数据库。

重算指标需要 pandas、numpy 和 scikit-learn；图表使用 altair。重新调用检测器还需要原项目的依赖、模型文件及本地配置。

In [1]:
from pathlib import Path
from collections import Counter
import csv, hashlib, importlib, importlib.metadata, json, math, os, re, sqlite3, sys, time, tomllib
import xml.etree.ElementTree as ET
from zipfile import ZipFile
import numpy as np
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'data/E/after_scores.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('请先解压配套包，并从包内或项目的 notebooks 目录打开本 notebook。')
DATA_DIR = ROOT / 'data/E'
EXPORT_DIR = ROOT / 'analysis/E_notebook_export/results'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
DETECTORS = list('ABCD')
NS = {'s': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
D_RESPONSE_ATTEMPTS = 4
D_MAX_TOKENS = 4096

def show_table(frame, digits=4):
    print(frame.to_string(index=False, float_format=lambda v: f'{v:.{digits}f}'))

print('找到配套数据。默认模式：只复核已有成绩，不重新调用检测器。')


找到配套数据。默认模式：只复核已有成绩，不重新调用检测器。


## 2. 标签、分数和三个指标

- `label/gold=0`：人类原文；`label/gold=1`：AI 来源的文本。改写后仍保留 AI 标签。
- `prob_ai`：检测器返回的 AI 分数，越大越像 AI。它用于排序；特别是 D 的分数是 LLM 给出的判断分数，不能直接当作校准后的真实概率。
- **AUROC**：AI 文本的分数排在人类文本前面的能力，计算时需要两类文本和连续分数。1 表示完全区分，0.5 对应随机排序的水平；不是“正确率”。本实验不为了提高成绩而翻转低于 0.5 的分数。
- **F1**：本次统一以 AI 为正类，`score >= 0.5` 判为 AI，计算 `average="binary"`。公式为 `2TP/(2TP+FP+FN)`。历史汇总中的 F1 保留成员原值，未在此重新统一其历史算法。
- **TPR@1%FPR**：在把人类误报成 AI 的比例不超过 1% 时，可检出的 AI 比例。取实测 ROC 点中 `FPR <= 0.01` 的最高 TPR，不做插值。150 篇人类中最多误报 1 篇才能满足该条件；该阈值不一定是 0.5。

In [2]:
def digest_bytes(value):
    return hashlib.sha256(value).hexdigest()

def text_digest(value):
    return digest_bytes(value.encode("utf-8"))

def file_digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def read_xlsx(path):
    """Read the first worksheet; no extra Excel-reading package is required."""
    with ZipFile(path) as z:
        shared = []
        if "xl/sharedStrings.xml" in z.namelist():
            shared = ["".join(n.itertext()) for n in ET.fromstring(
                z.read("xl/sharedStrings.xml")).findall("s:si", NS)]
        book = ET.fromstring(z.read("xl/workbook.xml"))
        first = book.find("s:sheets/s:sheet", NS)
        rel_id = first.attrib["{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"]
        rels = ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))
        target = next(r.attrib["Target"] for r in rels if r.attrib["Id"] == rel_id)
        if target.startswith("/"):
            target = target.lstrip("/")
        else:
            target = "xl/" + target
        grid = []
        for row in ET.fromstring(z.read(target)).findall("s:sheetData/s:row", NS):
            cells = {}
            for c in row.findall("s:c", NS):
                column = re.match(r"[A-Z]+", c.attrib["r"])[0]
                index = 0
                for char in column:
                    index = index * 26 + ord(char) - 64
                if c.find("s:f", NS) is not None:
                    raise ValueError("Input contains a formula; use fixed text/labels only.")
                v = c.find("s:v", NS)
                if c.attrib.get("t") == "s":
                    value = shared[int(v.text)]
                elif c.attrib.get("t") == "inlineStr":
                    value = "".join(c.find("s:is", NS).itertext())
                else:
                    value = v.text if v is not None else ""
                cells[index - 1] = value
            if any(cells.values()):
                grid.append([cells.get(i, "") for i in range(max(cells) + 1)])
    if not grid:
        raise ValueError("Empty input workbook")
    headers = [str(x).strip() for x in grid[0]]
    if len(set(headers)) != len(headers):
        raise ValueError("Duplicate column names")
    result = []
    for row in grid[1:]:
        obj = {h: str(row[i]) if i < len(row) else "" for i, h in enumerate(headers)}
        result.append(obj)
    return result

def validate_score(value):
    score = float(value)
    if not math.isfinite(score) or not 0 <= score <= 1:
        raise ValueError("Detector returned an invalid AI score")
    return score

def calculate_metrics(gold, scores):
    from sklearn.metrics import f1_score, roc_auc_score, roc_curve
    if set(gold) != {0, 1}:
        raise ValueError("AUROC requires both true classes")
    fpr, tpr, _ = roc_curve(gold, scores)
    return {"auroc": float(roc_auc_score(gold, scores)),
            "f1_binary_at_0_5": float(f1_score(gold, [int(v >= .5) for v in scores], average="binary", pos_label=1, zero_division=0)),
            "tpr_at_1fpr": max(float(t) for f, t in zip(fpr, tpr) if f <= .01 + 1e-12)}


## 3. 检查原始数据和改写测试集

原始 AI 文本由 D 提供，包含 DeepSeek Chat 和 DeepSeek Reasoner 两种来源。改写文本由 AI 辅助完成，再整理成固定测试集；这不是“人工从零创作的人类文章”。

299 篇 AI 中，100 篇采用全文改写，剩余 199 篇以开头改述为主，部分结合句子拆分、合并、词语调整和段落重排。因此，“299 篇都改过”不表示每篇的每一句都完全重写。

以下代码核对每个 UID：原文对应关系、人类文本是否保持不变、AI 文本是否确实改动，以及真实标签是否正确。原始文件中的 Human/AI 先在内存中映射为 0/1，不修改源文件。

In [3]:
original = pd.read_csv(DATA_DIR / 'P7_new_generators.csv', encoding='utf-8-sig', dtype=str, keep_default_na=False)
original['label'] = original.label.map({'Human': '0', 'AI': '1', '0': '0', '1': '1'})
assert original.label.notna().all()
after = pd.DataFrame(read_xlsx(DATA_DIR / 'attack_after_449.xlsx'))
metadata = json.loads((DATA_DIR / 'run_metadata.json').read_text(encoding='utf-8-sig'))
assert len(original) == len(after) == 449
assert original.uid.is_unique and after.uid.is_unique
original_index = original.set_index('uid')
after['original_uid'] = after['uid'].str.removeprefix('paraphrased_')
assert set(after.original_uid) == set(original.uid)
assert set(after.label) == {'0', '1'}
for row in after.to_dict('records'):
    source = original_index.loc[row['original_uid']]
    assert row['label'] == source['label']
    assert row['prompt_id'] == source['prompt_id']
    assert row['text'].strip()
    if row['label'] == '0':
        assert row['condition'] == 'human' and row['text'] == source['text']
    else:
        assert row['condition'] == 'after' and row['text'] != source['text']
        assert row['original_source_model'] == source['source_model']
assert file_digest(DATA_DIR / 'attack_after_449.xlsx') == metadata['after_sha256']
human_n = int((after.label == '0').sum())
ai_n = int((after.label == '1').sum())
assert (human_n, ai_n) == (150, 299)
print(f'核对通过：总计 {len(after)} 篇；人类原文 {human_n} 篇；AI 改写文本 {ai_n} 篇。')
print('\n原始 AI 来源：')
show_table(original.loc[original.label == '1', 'source_model'].value_counts().rename_axis('source_model').reset_index(name='n'))
print('\n改写方法（组合方法保留原记录）：')
methods = after.loc[after.label == '1', 'rewrite_method'].value_counts().rename_axis('rewrite_method').reset_index(name='n')
show_table(methods)
full_n = int((after.loc[after.label == '1', 'rewrite_method'] == 'full_passage_paraphrase').sum())
assert full_n == 100
print(f'全文改写 {full_n} 篇；以开头改述及组合方法为主 {ai_n - full_n} 篇。')


核对通过：总计 449 篇；人类原文 150 篇；AI 改写文本 299 篇。

原始 AI 来源：
     source_model   n
    deepseek-chat 150
deepseek-reasoner 149

改写方法（组合方法保留原记录）：
                                                                       rewrite_method   n
                                                                     opening_rephrase 104
                                                              full_passage_paraphrase 100
                                               opening_rephrase;paragraph_restructure  27
                                                      opening_rephrase;sentence_merge  19
                                                      opening_rephrase;sentence_split  11
                                opening_rephrase;sentence_merge;paragraph_restructure  10
                                     opening_rephrase;lexical_or_connective_variation  10
               opening_rephrase;lexical_or_connective_variation;paragraph_restructure  10
                      opening_rephrase;lexical_or_conne

## 4. 读取真实的逐条检测记录

`after_scores.csv` 是实际运行保存的记录，不是示例分数。每个检测器需要完整覆盖同一份 449 条测试文本；四个检测器合计 1,796 条成功记录。
检测器只接收 `text`，不会收到真实标签、`source_model` 或“改写后”等提示信息。下面逐条核对分数、标签以及文本 SHA-256，防止错用其他文件的成绩。

In [4]:
scores = pd.read_csv(DATA_DIR / 'after_scores.csv', encoding='utf-8-sig', dtype=str, keep_default_na=False)
assert len(scores) == 4 * len(after) == 1796
assert not scores.duplicated(['detector', 'uid']).any()
assert set(scores.detector) == set(DETECTORS)
assert set(scores.status) == {'ok'}
scores['prob_ai'] = pd.to_numeric(scores['prob_ai'], errors='raise')
scores['gold'] = pd.to_numeric(scores['gold'], errors='raise').astype(int)
assert np.isfinite(scores.prob_ai).all() and scores.prob_ai.between(0, 1).all()
target = after.set_index('uid')
for row in scores.to_dict('records'):
    record = target.loc[row['uid']]
    assert row['gold'] == int(record['label'])
    assert row['condition'] == record['condition']
    assert row['text_sha256'] == text_digest(record['text'])
    assert int(row['pred_at_0_5']) == int(row['prob_ai'] >= 0.5)
for detector in DETECTORS:
    assert set(scores.loc[scores.detector == detector, 'uid']) == set(after.uid)
coverage = scores.groupby('detector').agg(n=('uid', 'size'), ai_n=('gold', 'sum')).reset_index()
coverage['human_n'] = coverage.n - coverage.ai_n
print('逐条检测记录核对通过：')
show_table(coverage[['detector', 'n', 'human_n', 'ai_n']])


逐条检测记录核对通过：
detector   n  human_n  ai_n
       A 449      150   299
       B 449      150   299
       C 449      150   299
       D 449      150   299


## 5. 从逐条分数重新计算攻击后指标

这一步只是计算统计量，不会重新运行模型或访问 DeepSeek。最后把重算结果与原检测程序保存的 `after_metrics_recorded.csv` 逐项对照。

In [5]:
metric_rows = []
for detector in DETECTORS:
    part = scores[scores.detector == detector]
    values = calculate_metrics(part.gold.tolist(), part.prob_ai.tolist())
    metric_rows.append({'detector': detector, 'n': len(part),
                        'human_n': int((part.gold == 0).sum()),
                        'ai_n': int((part.gold == 1).sum()), **values})
metrics = pd.DataFrame(metric_rows)
recorded = pd.read_csv(DATA_DIR / 'after_metrics_recorded.csv', encoding='utf-8-sig').set_index('detector')
for row in metric_rows:
    for key, value in row.items():
        if key != 'detector':
            assert math.isclose(float(value), float(recorded.loc[row['detector'], key]), rel_tol=0, abs_tol=1e-12)
metrics.to_csv(EXPORT_DIR.parent / 'after_metrics_recomputed.csv', index=False, encoding='utf-8-sig')
print('重算结果与原始程序输出一致（绝对误差不超过 1e-12）：')
show_table(metrics)


重算结果与原始程序输出一致（绝对误差不超过 1e-12）：
detector   n  human_n  ai_n  auroc  f1_binary_at_0_5  tpr_at_1fpr
       A 449      150   299 0.7345            0.7877       0.0167
       B 449      150   299 0.2841            0.0189       0.0000
       C 449      150   299 0.5572            0.5000       0.1204
       D 449      150   299 0.9692            0.9233       0.5084


## 6. 汇总历史成绩，并说明比较的边界

HC3、MAGE、RAID、新生成器的主结果沿用成员上传后整理的成绩，不重新训练，也不重新计算历史 F1。`auroc_reported.csv` 是已经整理好的 16 条主结果，保留用户提交时的数值精度。

攻击前使用成员历史成绩，攻击后使用本次新检测的 449 条结果。历史逐条文件显示 A、B、C、D 的新生成器样本数并不完全相同；此外，D 本次将回复 token 上限由原接口的 400 调整为 4096，并对失败请求做有限次重试。因此，这是一项历史成绩与攻击后成绩的并列比较，不能把所有差异都当作“纯改写导致的变化”。

In [6]:
auroc = pd.read_csv(DATA_DIR / 'auroc_reported.csv', encoding='utf-8-sig', dtype=str, keep_default_na=False)
expected_header = ['dataset', 'detector', 'auroc', 'f1', 'tpr_at_1fpr']
assert list(auroc.columns) == expected_header
assert len(auroc) == 16 and not auroc.duplicated(['dataset', 'detector']).any()
assert set(auroc.dataset) == {'HC3', 'MAGE', 'RAID', 'New generators'}
historical_coverage = []
for detector in DETECTORS:
    member = pd.read_csv(DATA_DIR / 'member_results' / f'P7_{detector}_preds.csv', dtype=str, keep_default_na=False, encoding='utf-8-sig')
    is_new = member.dataset.str.lower().str.replace('_', ' ', regex=False).isin(['new generators', 'newgen'])
    part = member[is_new]
    historical_coverage.append({'detector': detector, 'before_n': len(part),
        'human_n_before': int((part.gold == '0').sum()), 'ai_n_before': int((part.gold == '1').sum()), 'after_n': 449})
print('历史新生成器逐条记录的覆盖数量：')
show_table(pd.DataFrame(historical_coverage))
before_values = auroc[auroc.dataset == 'New generators'][['detector', 'auroc']].rename(columns={'auroc': 'auroc_before'})
comparison = before_values.merge(metrics[['detector', 'auroc']], on='detector', validate='one_to_one').rename(columns={'auroc': 'auroc_after'})
comparison['auroc_before'] = comparison.auroc_before.astype(float)
comparison['delta_after_minus_before'] = comparison.auroc_after - comparison.auroc_before
print('\nAUROC 并列比较（负数表示当前记录中攻击后较低，不代表已排除其他影响因素）：')
show_table(comparison)


历史新生成器逐条记录的覆盖数量：
detector  before_n  human_n_before  ai_n_before  after_n
       A       420             121          299      449
       B       449             150          299      449
       C       449             150          299      449
       D       445             149          296      449

AUROC 并列比较（负数表示当前记录中攻击后较低，不代表已排除其他影响因素）：
detector  auroc_before  auroc_after  delta_after_minus_before
       A        0.7600       0.7345                   -0.0255
       B        0.4700       0.2841                   -0.1859
       C        0.6100       0.5572                   -0.0528
       D        0.9500       0.9692                    0.0192


## 7. 绘制攻击前后对比图

纵轴固定为 0–1，完整显示 B 的低分结果。该图表示上节的历史/当前成绩比较；颜色区分前后阶段，不表示人类/AI 文本类型。

In [7]:
import altair as alt
chart_data = comparison[['detector', 'auroc_before', 'auroc_after']].melt(
    id_vars='detector', var_name='stage', value_name='auroc')
chart_data['stage'] = chart_data.stage.map({'auroc_before': 'Historical before', 'auroc_after': 'After rewriting'})
chart = alt.Chart(chart_data).mark_bar().encode(
    x=alt.X('detector:N', title='Detector', sort=DETECTORS),
    xOffset=alt.XOffset('stage:N', sort=['Historical before', 'After rewriting']),
    y=alt.Y('auroc:Q', title='AUROC', scale=alt.Scale(domain=[0, 1])),
    color=alt.Color('stage:N', title='Evaluation', scale=alt.Scale(
        domain=['Historical before', 'After rewriting'], range=['#777777', '#e88930'])),
    tooltip=['detector:N', 'stage:N', alt.Tooltip('auroc:Q', format='.4f')]
).properties(width=560, height=300, title='Historical baseline vs. after rewriting')
CHART_SPEC = chart.to_dict()
chart.save(EXPORT_DIR.parent / 'P7_E_attack_auroc.html')
(EXPORT_DIR.parent / 'P7_E_attack_auroc.vl.json').write_text(json.dumps(CHART_SPEC, ensure_ascii=False), encoding='utf-8')
print('已生成对比图；HTML 图表可用浏览器查看（加载交互组件需要联网）。')
try:
    from IPython.display import display
except ImportError:
    pass
else:
    display(chart)


已生成对比图；HTML 图表可用浏览器查看（加载交互组件需要联网）。


AUROC chart: historical baseline vs. after rewriting; y-axis 0–1.

## 8. 核对演示配对，导出四个交付文件

`attack_pairs.csv` 的四组原文与改写来自实际测试文本，而不是重新编造的演示文章。这里逐字核对，再导出到独立目录。

主结果 `auroc.csv` 原值保留。攻击对比中，`auroc_before` 保留历史汇总值；攻击后数值从 1,796 条记录重算，保存完整精度，展示时可保留两位小数。两份 CSV 的小数位数可能与手动四舍五入的提交版本不同，但不会更改检测器的原始分数。

In [8]:
pairs = pd.read_csv(DATA_DIR / 'attack_pairs_reported.csv', encoding='utf-8-sig', dtype=str, keep_default_na=False)
assert {'id', 'uid', 'prompt_id', 'original', 'attacked', 'attack_type'} <= set(pairs.columns)
assert 3 <= len(pairs) <= 5 and pairs.id.is_unique and pairs.uid.is_unique
after_index = after.set_index('original_uid')
for pair in pairs.to_dict('records'):
    assert pair['original'] == original_index.loc[pair['uid'], 'text']
    assert pair['attacked'] == after_index.loc[pair['uid'], 'text']
    assert pair['prompt_id'] == original_index.loc[pair['uid'], 'prompt_id']
    assert original_index.loc[pair['uid'], 'label'] == '1'
attack = before_values.merge(metrics, on='detector', validate='one_to_one').rename(columns={
    'auroc': 'auroc_after', 'f1_binary_at_0_5': 'f1_binary_at_0_5_after',
    'tpr_at_1fpr': 'tpr_at_1fpr_after', 'n': 'total',
    'human_n': 'human_Original', 'ai_n': 'ai_Rewritten'})
attack = attack[['detector', 'auroc_before', 'auroc_after', 'f1_binary_at_0_5_after',
                 'tpr_at_1fpr_after', 'total', 'human_Original', 'ai_Rewritten']]
auroc.to_csv(EXPORT_DIR / 'auroc.csv', index=False, encoding='utf-8-sig')
attack.to_csv(EXPORT_DIR / 'attack.csv', index=False, encoding='utf-8-sig')
pairs.to_csv(EXPORT_DIR / 'attack_pairs.csv', index=False, encoding='utf-8-sig')
indexed = comparison.set_index('detector')
d_result = metrics.set_index('detector').loc['D']
findings = [
    '按现有结果对比，' + '、'.join(f'{d}的AUROC由{indexed.loc[d, "auroc_before"]:.2f}变为{indexed.loc[d, "auroc_after"]:.2f}' for d in 'ABC') + '；其中B的降幅最大。',
    '攻击后，B的TPR@1%FPR为0，表示在本次测试中，将人类文本误报为AI的比例限制在1%以内时，未能检出AI文本。',
    f'攻击后包含150篇未改写的人类文本和299篇AI改写文本；D表现最好，AUROC为{d_result.auroc:.2f}，F1（二分类，阈值0.5）为{d_result.f1_binary_at_0_5:.2f}，TPR@1%FPR为{d_result.tpr_at_1fpr:.2f}。',
    '攻击前采用成员原有结果，攻击后采用本次重新检测结果；样本数量和运行设置未完全统一，不能把全部差异都归因于改写。'
]
(EXPORT_DIR / 'findings.txt').write_text('\n'.join(findings) + '\n', encoding='utf-8-sig')
for name in ['auroc.csv', 'attack.csv', 'attack_pairs.csv']:
    check = pd.read_csv(EXPORT_DIR / name, encoding='utf-8-sig')
    print(f'{name}: {len(check)} 条数据，CSV 可正常读取。')
print('findings.txt: 4 条结论。')
print('输出位置：analysis/E_notebook_export/results；尚未覆盖原来的 results。')


auroc.csv: 16 条数据，CSV 可正常读取。
attack.csv: 4 条数据，CSV 可正常读取。
attack_pairs.csv: 4 条数据，CSV 可正常读取。
findings.txt: 4 条结论。
输出位置：analysis/E_notebook_export/results；尚未覆盖原来的 results。


## 9. 可选：重新运行真实检测器的完整代码

**本次已经检测完成，不需要执行新的检测。** 下列代码提供实验方法的可复现入口，默认 `RUN_NEW_DETECTION=False`，不会发出请求。

A 调用 `predict()`；B 调用 `predict()`；C 以 8 篇为一批调用 `score_texts()` 获取整篇分数；D 使用成员的模型、提示词和规范化函数，`max_tokens=4096`，最多 4 次外层尝试，保留 SDK 原有重试，接口失败不填 0。D 的推理/思考设置沿用服务端默认，因此后续模型或服务默认值变化可能导致不同结果。

重新运行需在本地配置 API key，D 会产生调用费用；模型首次运行可能下载。不要将 key 写入 notebook 或上传。新检测使用独立目录，不会写入原先已完成的 `progress.sqlite3`。原有 1,796 条成绩始终用于上面的正式复核部分，不会被新运行静默替换。

In [9]:
def load_local_settings(project):
    """Never print, copy, or write credentials to output files."""
    merged = {}
    for path in (Path.home() / ".streamlit/secrets.toml", project / ".streamlit/secrets.toml"):
        if path.exists():
            try:
                settings = tomllib.loads(path.read_text(encoding="utf-8-sig"))
                merged.update({k: v for k, v in settings.items() if isinstance(v, str)})
            except tomllib.TOMLDecodeError:
                raise ValueError("The local secrets.toml is not valid TOML; its contents are not shown.") from None
    for key, value in merged.items():
        os.environ.setdefault(key, value)
    try:
        from dotenv import load_dotenv
        load_dotenv(project / ".env", override=False)
    except ImportError:
        pass
    mode = os.environ.get("P7_D_PROMPT", "few_shot")
    if mode == "few_shot" and not os.environ.get("P7_D_FEWSHOT"):
        candidates = [project / "data/fewshot_examples.json",
                      project / "D_others/D to ABC/fewshot_examples.json",
                      project / "D_others/D  to F/fewshot_examples.json"]
        found = next((p for p in candidates if p.is_file()), None)
        if found:
            os.environ["P7_D_FEWSHOT"] = str(found)
    return mode

def safe_error(exc):
    text = f"{type(exc).__name__}: {exc}"
    for key in ("DEEPSEEK_API_KEY", "P7_API_KEY", "OPENAI_API_KEY"):
        secret = os.environ.get(key)
        if secret:
            text = text.replace(secret, "[redacted]")
    return re.sub(r"\bsk-[A-Za-z0-9_-]+", "[redacted]", text)[:1200]

def record(conn, detector, task, status, score=None, native_label="", elapsed=0.0, error=""):
    conn.execute("INSERT OR REPLACE INTO scores VALUES (?,?,?,?,?,?,?,?,?)", (
        detector, task["uid"], task["text_sha256"], status, score, native_label,
        elapsed, error, time.strftime("%Y-%m-%dT%H:%M:%S")))
    conn.commit()

def request_D_json(module,text):
    """Use the member's exact prompt/model with a sufficient, fixed reply budget."""
    cleaned=(module.normalize_text(text) if module.NORMALIZE else text.strip())[:module.MAX_CHARS]
    response=module._get_client().chat.completions.create(
        model=module.MODEL,temperature=0,max_tokens=D_MAX_TOKENS,
        response_format={"type":"json_object"},
        messages=[{"role":"system","content":module.SYSTEM_PROMPT},
                  {"role":"user","content":module._build_user_prompt(cleaned,module.PROMPT_MODE)}])
    if not response.choices:
        raise RuntimeError("Empty response: no choices")
    choice=response.choices[0]
    finish=getattr(choice,"finish_reason",None)
    content=(choice.message.content or "").strip()
    usage=getattr(response,"usage",None)
    completion_tokens=getattr(usage,"completion_tokens",None)
    if not content or finish=="length":
        print(f"[D] reply diagnostics: finish={finish}, content_chars={len(content)}, completion_tokens={completion_tokens}, max_tokens={D_MAX_TOKENS}",flush=True)
    if finish=="length":
        raise RuntimeError(f"Truncated response at max_tokens={D_MAX_TOKENS}")
    if not content:
        raise RuntimeError(f"Empty response (finish={finish}, completion_tokens={completion_tokens})")
    result=json.loads(content)
    value=result.get("ai_score") if isinstance(result,dict) else None
    if isinstance(value,bool) or not isinstance(value,(int,float)) or not math.isfinite(value) or not 0<=value<=100:
        raise ValueError("D returned an invalid ai_score")
    return {"ai_score":value,"reason":str(result.get("reason","")),"finish_reason":finish,
            "completion_tokens":completion_tokens}

def judge_with_response_retries(module, text, attempts=D_RESPONSE_ATTEMPTS):
    """Bounded identical-request retries; never score partial JSON or reasoning text."""
    for attempt in range(1, attempts + 1):
        try:
            return request_D_json(module,text)
        except Exception as exc:
            status = getattr(exc,"status_code",None)
            if status in (400,401,402,403,404) or type(exc).__name__=="BalanceError":
                raise
            transient = (isinstance(exc,(json.JSONDecodeError,TimeoutError,ConnectionError))
                         or (isinstance(exc,RuntimeError) and any(s in str(exc).lower() for s in ("empty response","truncated response")))
                         or status==429 or (isinstance(status,int) and status>=500)
                         or type(exc).__name__ in {"APITimeoutError","APIConnectionError","RateLimitError","InternalServerError"})
            if not transient or attempt==attempts:
                raise
            delay=min(2**(attempt-1),8)
            print(f"[D] incomplete/temporary reply; retry {attempt+1}/{attempts} in {delay}s",flush=True)
            time.sleep(delay)

def run_detector(conn, project, detector, tasks, limit, batch_size):
    selected = tasks[:limit] if limit else tasks
    completed = {uid: sha for uid, sha in conn.execute(
        "SELECT uid,text_sha256 FROM scores WHERE detector=? AND status='ok'", (detector,))}
    pending = [r for r in selected if completed.get(r["uid"]) != r["text_sha256"]]
    print(f"\n[{detector}] selected={len(selected)}, cached={len(selected)-len(pending)}, remaining={len(pending)}", flush=True)
    if not pending:
        return
    # Direct member interfaces only. Never use the front end's mock fallback.
    module = importlib.import_module(f"P7_{detector}_predict")
    failures = 0
    started = time.monotonic()
    chunks = [pending[i:i + batch_size] for i in range(0, len(pending), batch_size)] if detector == "C" else [[r] for r in pending]
    for chunk in chunks:
        if detector == "C":
            tick = time.monotonic()
            try:
                result = module.score_texts([r["text"] for r in chunk], batch_size=batch_size)
                if len(result) != len(chunk):
                    raise ValueError("C returned the wrong number of scores")
                result = [validate_score(v) for v in result]
                per_text = (time.monotonic() - tick) / len(chunk)
                for task, score in zip(chunk, result):
                    record(conn, detector, task, "ok", score, "AI" if score >= .5 else "Human", per_text)
                failures = 0
            except Exception as exc:
                for task in chunk:
                    record(conn, detector, task, "error", error=safe_error(exc))
                failures += len(chunk)
                print(f"[{detector}] ERROR: {safe_error(exc)}", flush=True)
        else:
            task = chunk[0]
            tick = time.monotonic()
            try:
                if detector == "D":
                    r = judge_with_response_retries(module,task["text"])
                    score = validate_score(r["ai_score"] / 100)
                    native = "AI" if score >= module.THRESHOLD else "Human"
                else:
                    r = module.predict(task["text"])
                    if not isinstance(r, dict) or "prob_ai" not in r:
                        raise ValueError("Detector returned no score")
                    if (r.get("extra") or {}).get("note"):
                        raise ValueError("Detector marked passage unusable: " + str(r["extra"]["note"]))
                    score = validate_score(r["prob_ai"])
                    native = str(r.get("label", ""))
                record(conn, detector, task, "ok", score, native, time.monotonic() - tick)
                failures = 0
            except Exception as exc:
                record(conn, detector, task, "error", error=safe_error(exc), elapsed=time.monotonic()-tick)
                failures += 1
                print(f"[{detector}] {task['uid']} ERROR: {safe_error(exc)}", flush=True)
                if getattr(exc, "status_code", None) in (401, 402, 403) or type(exc).__name__ == "BalanceError":
                    raise RuntimeError("API authentication, access, or balance error. Progress was saved.") from None
        ok_ids = {r[0] for r in conn.execute("SELECT uid FROM scores WHERE detector=? AND status='ok'", (detector,))}
        total = sum(r["uid"] in ok_ids for r in tasks)
        print(f"[{detector}] saved {total}/{len(tasks)} | {time.monotonic()-started:.1f}s this session", flush=True)
        if failures >= 3:
            raise RuntimeError(f"{detector}: repeated failures. Stopped instead of filling failures with zero.")


In [10]:
RUN_NEW_DETECTION = False
# 仅在确实需要重新检测时修改。示例原项目通常位于桌面的 AI-Detector-main。
PROJECT_DIR = ROOT if (ROOT / 'P7_A_predict.py').is_file() else Path.home() / 'Desktop/AI-Detector-main'
NEW_RUN_DIR = ROOT / 'analysis/E_notebook_new_detection'

def open_new_run_cache(path, fingerprint):
    conn = sqlite3.connect(path)
    conn.execute('CREATE TABLE IF NOT EXISTS metadata (key TEXT PRIMARY KEY, value TEXT NOT NULL)')
    conn.execute("""CREATE TABLE IF NOT EXISTS scores (
        detector TEXT, uid TEXT, text_sha256 TEXT NOT NULL, status TEXT NOT NULL,
        prob_ai REAL, native_label TEXT, elapsed_seconds REAL, error TEXT,
        updated_at TEXT, PRIMARY KEY(detector,uid))""")
    serial = json.dumps(fingerprint, sort_keys=True, ensure_ascii=False)
    saved = conn.execute("SELECT value FROM metadata WHERE key='fingerprint'").fetchone()
    if saved and saved[0] != serial:
        conn.close()
        raise ValueError('新检测的输入、代码、设置或依赖发生变化；请使用新的 NEW_RUN_DIR，勿混合成绩。')
    conn.execute('INSERT OR IGNORE INTO metadata VALUES (?, ?)', ('fingerprint', serial))
    conn.commit()
    return conn

def run_optional_new_detection():
    project = PROJECT_DIR.resolve()
    for d in DETECTORS:
        if not (project / f'P7_{d}_predict.py').is_file():
            raise FileNotFoundError(f'请确认 PROJECT_DIR 和 P7_{d}_predict.py 的位置。')
    mode = load_local_settings(project)
    if mode != 'few_shot':
        raise ValueError('本次实验使用 few_shot；当前设置不同，请先核对。')
    if not (os.environ.get('P7_API_KEY') or os.environ.get('DEEPSEEK_API_KEY')):
        raise ValueError('请在本地配置 D 的 API key，勿写进 notebook。')
    shot_path = Path(os.environ.get('P7_D_FEWSHOT', ''))
    if not shot_path.is_file():
        raise ValueError('缺少真实 few-shot 示例文件。')
    shots = json.loads(shot_path.read_text(encoding='utf-8-sig'))
    if not shots or any(str(s.get('qid', '')) in set(after.prompt_id) for s in shots if s.get('qid')):
        raise ValueError('few-shot 示例为空或与测试题目重叠。')
    if file_digest(shot_path) != metadata['settings']['fewshot_sha256']:
        raise ValueError('few-shot 示例与原实验不一致。')
    settings = {key: os.environ.get(key, default) for key, default in {
        'P7_D_MODEL': 'deepseek-flash', 'P7_D_PROMPT': 'few_shot',
        'P7_D_THRESHOLD': '0.50', 'P7_D_NORMALIZE': '1', 'P7_B_MODEL': 'gpt2',
        'P7_C_MODEL': 'Hanxi79/p7-detector', 'P7_B_CALIBRATOR_REPO': 'captainfan/p7-gpt2-gltr-detector'}.items()}
    settings['base_url_sha256'] = text_digest(os.environ.get('P7_BASE_URL', 'https://api.deepseek.com'))
    settings['fewshot_sha256'] = file_digest(shot_path)
    if os.environ.get('P7_B_CALIBRATOR'):
        settings['B_local_classifier_sha256'] = file_digest(Path(os.environ['P7_B_CALIBRATOR']))
    if settings != metadata['settings']:
        raise ValueError('当前检测器设置与原实验不一致，先核对模型及配置。')
    module_paths = [project / f'P7_{d}_predict.py' for d in DETECTORS] + [project / 'models/P7_A_model.pkl']
    current_files = {str(p.relative_to(project)): file_digest(p) for p in module_paths if p.is_file()}
    if current_files != metadata['detector_files']:
        raise ValueError('检测器或 A 模型文件已变化，先核对版本。')
    fingerprint = {'after_sha256': file_digest(DATA_DIR / 'attack_after_449.xlsx'),
        'detector_files': current_files, 'settings': settings,
        'versions': {p: importlib.metadata.version(p) for p in ['numpy', 'scikit-learn', 'xgboost', 'torch', 'transformers', 'openai']},
        'D_max_tokens': D_MAX_TOKENS, 'D_response_attempts': D_RESPONSE_ATTEMPTS,
        'code_sha256': hashlib.sha256(run_detector.__code__.co_code + request_D_json.__code__.co_code + judge_with_response_retries.__code__.co_code).hexdigest()}
    tasks = [dict(r, text_sha256=text_digest(r['text'])) for r in after.to_dict('records')]
    NEW_RUN_DIR.mkdir(parents=True, exist_ok=True)
    conn = open_new_run_cache(NEW_RUN_DIR / 'progress.sqlite3', fingerprint)
    old_cwd = Path.cwd()
    old_path = list(sys.path)
    try:
        sys.path.insert(0, str(project))
        os.chdir(project)
        if any(f'P7_{d}_predict' in sys.modules for d in DETECTORS):
            raise RuntimeError('请重启 notebook 内核后再重新检测，防止重复使用旧模块。')
        for detector in DETECTORS:
            run_detector(conn, project, detector, tasks, limit=0, batch_size=8)
    finally:
        os.chdir(old_cwd)
        sys.path[:] = old_path
        saved = pd.read_sql_query('SELECT * FROM scores ORDER BY detector, uid', conn)
        saved.to_csv(NEW_RUN_DIR / 'new_scores.csv', index=False, encoding='utf-8-sig')
        (NEW_RUN_DIR / 'metadata.json').write_text(json.dumps(fingerprint, ensure_ascii=False, indent=2), encoding='utf-8')
        conn.close()
    complete = len(saved) == 4 * len(after) and set(saved.status) == {'ok'}
    if complete:
        new_rows = []
        for d in DETECTORS:
            subset = saved[saved.detector == d]
            labels = [int(target.loc[uid, 'label']) for uid in subset.uid]
            new_rows.append({'detector': d, **calculate_metrics(labels, subset.prob_ai.tolist())})
        pd.DataFrame(new_rows).to_csv(NEW_RUN_DIR / 'new_metrics.csv', index=False, encoding='utf-8-sig')
        print('新检测完成，已保存独立结果，未替换原实验成绩。')
    else:
        print('新检测尚未完整完成；已有进度保留，未生成最终新指标。')

if RUN_NEW_DETECTION:
    run_optional_new_detection()
else:
    print('已跳过重新检测：本次仅复核原有真实成绩，没有发出 API 请求。')


已跳过重新检测：本次仅复核原有真实成绩，没有发出 API 请求。


## 10. 实验说明、限制与参考

- 本次没有对 A–D 重新训练；只是用现有接口打分。不能把 E 的任务描述成“训练四个模型”。
- 固定改写文本是实验输入，不在默认运行中再次随机生成，保证本地复核使用相同文本。299 篇改写强度不同，不能据此推断所有类型、所有强度攻击都具有相同效果。
- 原文来自 D 的固定测试集。旧汇总的四个新生成器样本量为 A=420、B=449、C=449、D=445；本次攻击后均为 449。只比较已报告成绩，不声称严格配对的因果效果。
- A/B 原模型加载曾出现 scikit-learn 版本不一致提示；D 的线上模型别名和服务端默认行为也可能变化。历史运行元信息及文件哈希已保留，未来重跑不保证逐条分数完全相同。
- 本次攻击后 F1 为 binary、AI 正类、阈值 0.5。不要把成员历史 F1 在没有核对的情况下统称为相同定义。
- `data/E/run_metadata.json` 记录原检测时依赖、模型和接口文件哈希，不含 API key。下面的默认复核依赖与模型推理依赖不同，Jupyter 只是打开 `.ipynb` 的工具。

参考：项目仓库 <https://github.com/ZiyingTeng/AI-Detector>；原始运行元信息和成员导出文件（随包保存）；scikit-learn 的 `roc_auc_score`、`f1_score`、`roc_curve` 实现。本文对 AUROC 的解释为正类/负类二分类情况。

In [11]:
print('原检测环境记录：')
show_table(pd.DataFrame([{'package': name, 'version': version} for name, version in metadata['versions'].items()]))
print('\nD 原运行设置：', metadata['settings']['P7_D_MODEL'], metadata['settings']['P7_D_PROMPT'],
      'max_tokens=', metadata['D_request_policy']['max_tokens'])
print('完成：输入核对、1,796 条成绩核对、指标重算、历史覆盖核对、演示配对核对及四个文件导出。')


原检测环境记录：
     package version
       numpy   2.5.4
scikit-learn   1.9.1
     xgboost   3.2.0
       torch  2.15.0
transformers  5.22.1
      openai  3.49.0

D 原运行设置： deepseek-flash few_shot max_tokens= 4096
完成：输入核对、1,796 条成绩核对、指标重算、历史覆盖核对、演示配对核对及四个文件导出。
